# Welsh ASR — XLS-R fine-tuning on Colab

Runs either in browser Colab or through the official Google Colab extension for
VS Code / Cursor. Both execute on the same free T4; the extension just keeps the
notebook in your repo instead of the browser.

Either way the runtime is a separate machine that cannot see your local files,
which is why cell 3 clones the repo.

**Secrets** are read from Colab Secrets when present, otherwise prompted for.
Nothing is written into the notebook. You need `GH_TOKEN` (repo scope),
`HF_TOKEN` (write) and `WANDB_API_KEY`.

**GPU:** browser -> Runtime -> Change runtime type -> T4.
Extension -> kernel picker (top right) -> Colab -> T4.

Target to beat: zero-shot Whisper-small, **WER 0.5987** on FLEURS Welsh test.

## 1. Confirm a GPU was actually allocated

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
# Free tier sometimes hands you a CPU-only runtime. If this errors, stop and
# retry later rather than training on CPU.

## 2. Install dependencies

In [ ]:
!pip install -q "transformers>=5.0" "datasets>=5.0" jiwer accelerate soundfile librosa wandb
# Colab ships transformers 4.x. This upgrade matters: 5.x renamed the flags
# train.py uses. If imports misbehave afterwards, Runtime -> Restart session.

## 3. Get the code

The repo is private, so cloning needs a GitHub token with `repo` scope stored in
Colab Secrets as `GH_TOKEN`. If you have made the repo public, the fallback
clone works without one.

In [ ]:
import os, getpass

def secret(name, prompt=None):
    """Fetch a secret from Colab Secrets, the environment, or an interactive prompt.

    google.colab.userdata only exists in the browser UI; under the VS Code /
    Cursor extension it is absent, so fall back to getpass, which also keeps the
    value out of the saved notebook.
    """
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return getpass.getpass(prompt or f"{name}: ")


import subprocess

REPO = "paarthN/welsh-asr"
DIR = "/content/welsh-asr"
tok = secret("GH_TOKEN", "GitHub token (repo scope): ")
url = f"https://{tok}@github.com/{REPO}.git"

# Clone on a fresh runtime, otherwise pull. Skipping outright when the
# directory exists would silently keep running whatever code was fetched
# during the previous session.
if os.path.exists(DIR):
    subprocess.run(["git", "-C", DIR, "remote", "set-url", "origin", url], check=True)
    subprocess.run(["git", "-C", DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, DIR], check=True)

%cd /content/welsh-asr
!git log --oneline -1

## 4. Secrets and W&B

In [ ]:
os.environ["HF_TOKEN"] = secret("HF_TOKEN", "Hugging Face token (write): ")
os.environ["WANDB_API_KEY"] = secret("WANDB_API_KEY", "W&B API key: ")
os.environ["WANDB_PROJECT"] = "welsh-asr"

import wandb; wandb.login()

## 5. Mount Drive for checkpoints

This is what makes a disconnect survivable. Each checkpoint is ~3.6GB (model
plus optimizer state) and `save_total_limit=2` keeps two, so budget ~7GB of the
free 15GB.

In [ ]:
# Drive gives crash-resilient checkpoints. It is available in browser Colab;
# in the VS Code extension it may not be, so fall back to the runtime's local
# disk. Local disk is wiped when the session ends, so if Drive is unavailable
# treat the run as needing to finish in one session.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/welsh-asr-xlsr"
except Exception as e:
    print(f"Drive unavailable ({type(e).__name__}); using local disk.")
    print("WARNING: checkpoints will NOT survive the session ending.")
    CKPT_DIR = "/content/welsh-asr-xlsr"

os.makedirs(CKPT_DIR, exist_ok=True)
print("checkpoints ->", CKPT_DIR)

## 6. Build the vocabulary and processor (CPU, seconds)

In [ ]:
!cd src && python prepare_data.py

## 7. Train

**Start at batch size 2.** Measured locally, batch 4 with 30s clips allocated
27GB in fp32; halved for fp16 that is still ~13.5GB of activations plus ~5GB of
model and optimizer state, which does not fit a 16GB T4.

Step DOWN a row only if CUDA reports OOM. Effective batch stays 16 throughout,
so the learning dynamics do not shift between rows.

| Try | `--batch-size` | `--grad-accum` | `--gradient-checkpointing` |
|---|---|---|---|
| 1 | 2 | 8 | off |
| 2 | 2 | 8 | on |
| 3 | 2 | 8 | on, plus `--max-duration 20` |
| 4 | 1 | 16 | on, plus `--max-duration 20` |

If try 1 leaves plenty of headroom in `nvidia-smi`, `--batch-size 4
--grad-accum 4 --gradient-checkpointing` will run faster.

`max_steps=2100` is about 12 epochs over the 2784 usable clips.

**Watch for:** loss falling from ~19 and continuing down; eval WER starting at
1.0 and dropping below 0.5 within a few evals. If eval WER is still 1.0 after
~1000 steps, stop — that is a vocab problem, not impatience.

In [ ]:
!cd src && python -u train.py \
    --output-dir "{CKPT_DIR}" \
    --max-steps 2100 \
    --batch-size 2 \
    --grad-accum 8 \
    --hub-model-id pnawani/welsh-asr-xlsr-300m \
    --push-to-hub

## 8. Resume after a disconnect

Re-run cells 1-6, then run this instead of cell 7. It picks up from the last
checkpoint on Drive.

In [ ]:
!cd src && python -u train.py \
    --output-dir "{CKPT_DIR}" \
    --max-steps 2100 \
    --batch-size 2 \
    --grad-accum 8 \
    --hub-model-id pnawani/welsh-asr-xlsr-300m \
    --push-to-hub \
    --resume

## 9. Evaluate the fine-tuned model on the test set

Writes `results/finetuned_results.json`. Download it and run
`src/error_analysis.py` locally against it plus the Whisper baseline.

In [ ]:
!cd src && python -u evaluate.py \
    --model "{CKPT_DIR}" \
    --out ../results/finetuned_results.json \
    --progress-every 50

In [ ]:
from google.colab import files
files.download("/content/welsh-asr/results/finetuned_results.json")